# 03. 상권 × 업종 과거 성과 근거 생성

2021Q1~2025Q4의 실제 매출 관측 조합을 기준으로 성장성, 안정성, 경쟁도, 개폐업 위험과 데이터 신뢰도를 집계한다. 미래 매출을 예측하지 않는다.

최근 4분기는 2025Q1~Q4, 비교 4분기는 2024Q1~Q4로 고정한다. 사용할 수 없는 성장률이나 최근 매출은 0으로 바꾸지 않고 NaN 및 원인 플래그로 보존한다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.area_industry_evidence import build_area_industry_evidence
PROJECT_ROOT

In [ ]:
result = build_area_industry_evidence(project_root=PROJECT_ROOT)
evidence = result.evidence
dictionary = result.dictionary
validation = result.validation
stale = result.stale_combinations

In [ ]:
summary = pd.Series({
    'rows_and_combinations': len(evidence),
    'areas': evidence['area_code'].nunique(),
    'industries': evidence['industry_code'].nunique(),
    'columns': len(evidence.columns),
    'feature_columns': int(dictionary['role'].eq('feature').sum()),
    'quality_columns': int(dictionary['role'].eq('quality').sum()),
    'grain_duplicate_rows': int(evidence.duplicated(['area_code', 'industry_code'], keep=False).sum()),
    'stale_combinations': int(evidence['stale_observation_flag'].sum()),
    'mean_data_reliability': evidence['data_reliability'].mean(),
})
summary

## 신뢰도와 관측 분기

신뢰도는 매출 coverage, 기간 내 continuity, 최근성, 점포 표본, area profile 신뢰도와 점포 데이터 coverage를 결합한다. 최근 관측이 4분기 이상 오래된 조합은 stale로 분류한다.

In [ ]:
evidence['reliability_grade'].value_counts().sort_index().rename_axis('grade').to_frame('combinations')

In [ ]:
evidence['observed_quarter_count'].value_counts().sort_index().rename_axis('observed_quarters').to_frame('combinations')

## 검증과 분포

성장률은 1%/99% 분위수 및 안전 hard bound로 clipping하고, 변동계수·장기 추세·점포당 매출도 clipping 여부를 별도 플래그로 남긴다.

In [ ]:
validation.loc[validation['record_type'].eq('global_check'), ['name', 'value', 'status', 'details']]

In [ ]:
metrics = [
    'latest_observed_sales', 'recent_4q_average_sales', 'yoy_growth_rate',
    'recent_4q_growth_rate', 'long_term_sales_trend_slope',
    'sales_coefficient_of_variation', 'decline_quarter_ratio',
    'recent_4q_average_sales_per_store', 'competition_intensity',
    'opening_rate', 'closing_rate', 'churn_rate', 'data_reliability',
]
evidence[metrics].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

## 최근 관측 중단 조합

CS200036 고시원을 포함해 최근 매출이 중단된 조합은 reliability가 낮아지며 별도 CSV에 저장한다.

In [ ]:
stale.head(30)

In [ ]:
evidence.loc[evidence['industry_code'].eq('CS200036'), [
    'area_code', 'industry_name', 'latest_observed_quarter', 'quarter_age',
    'observed_quarter_count', 'data_reliability', 'reliability_grade',
    'stale_observation_flag',
]]

산출물:

- `data/processed/area_industry_evidence.parquet`
- `outputs/tables/area_industry_evidence_dictionary.csv`
- `outputs/tables/area_industry_evidence_validation.csv`
- `outputs/tables/stale_industry_combinations.csv`